# Loss-term test — region GNN: full loss vs Huber + MMD only (seeds 6–8, α = 0.6)

**Why:** only the Huber and MMD terms are computed on the correction Δ; NT-Xent and the adversarial
term act on the latent only, and neither has been removed on the region graph. If the simpler
objective is **at least as good** on every axis, it replaces the published loss (a gain is not
required: every remaining term would then have a shown purpose).

**Arms** (`spancy_shift_spatial.py`; per seed both arms share the init and the sampler stream;
one shared `prepare()`):
- `full` — published loss (w_contrast 0.5, w_adv 0.3).
- `mmd-huber` — w_contrast 0, w_adv 0 (Huber 0.1 + MMD 1.0 remain; the removed terms' gradients are exactly 0).
- `full-noise`, `mmd-huber-noise` — Stage 1 + Gaussian noise matched per batch and marker to each arm's
  correction (`matched_noise`, verbatim from `spancy_shift_region_final.ipynb`; the full arm uses the
  same noise seeds as region_final's `region-noise`).

**Adoption rule (fixed 2026-10-13, before any result; mmd-huber vs full, paired per seed):**
1. kBET no worse: mean ΔkBET ≥ −0.011.
2. Biology no worse: mean Δsilhouette ≥ −0.002 and no seed < −0.005; no seed with more distorted
   markers; positive-population pass count not lower at any seed.
3. C1 (structured, not blur): kBET and silhouette above its own matched noise at every seed.
4. C2 (vs Stage 1 → MMD-ResNet): mean kBET ≥ 0.7016 and silhouette > 0.3288 at every seed.
5. Stability: kBET SD over seeds ≤ 2 × the full loss's SD.

**All five pass → adopt Huber + MMD; any fails → keep the published loss** (the Discussion then
reports the 3-seed differences).

**Order:** §1a pre-flight → §3 training (~90 min: 6 × 15 min) → §4–§6 metrics → §7 histograms (seed 6)
→ **§7c save (download the zip)** → §8 kBET on 12 layers (**≈ 4 h**, can run later: fresh runtime →
§0 install → §R restore → §8 → §9) → §9 verdict.
**Uploads:** `spancy_shift_spatial.py` (tidied version, branch `experiments/ntxent-mask`) and
`spancy_shift_region.py` (only for the verbatim helper imports). **Paste back §9.**

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
from google.colab import files
import os

NEEDED = ['spancy_shift_spatial.py', 'spancy_shift_region.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()
for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:28s} {os.path.getsize(f):>10,} bytes')
assert 'def _apply_shifts' in open('spancy_shift_spatial.py').read(), \
    'OLD spancy_shift_spatial.py — upload the tidied one from branch experiments/ntxent-mask'
print('Modules: OK')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import iqr as _iqr
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift_spatial as ss   # region GNN (primary method)
from spancy_shift_region import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    positive_population_table,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 1a. Prepare (Stage 1 + region graph, shared by all arms) + pre-flight

In [ ]:
o = adata.obs
prep = ss.prepare(adata, k_neighbors=15, bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC)
samp = pd.Categorical(o['sample_id'].astype(str)).codes
assert len(np.unique(prep.regions)) == 75, 'expected 75 (sample, scene) regions'
assert (samp[prep.adj.knn] == samp[:, None]).all(), 'region graph links different samples'
assert (prep.is_bimodal == marker_is_bimodal).all(), 'bimodal markers differ from §2'

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}
for g, pairs in GROUP_DEFS.items():
    for s, b in pairs:
        assert ((o['sample_id'] == s) & (o['batch_id'] == b)).any(), f'kBET group {g}: no cells for ({s}, {b})'
print('PRE-FLIGHT PASSED')

## 3. Train: full and mmd-huber × seeds 6–8 (~90 min) + matched-noise controls

In [ ]:
SEEDS = [6, 7, 8]
POP_SEEDS = SEEDS                 # rule 2 compares positive populations at every seed
ALPHA = 0.6
BASE = dict(n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512, temperature=0.07,
            w_recon=0.1, w_mmd=1.0, mmd_samples=256, grl_max=1.0)
ARMS = [('full', dict(w_contrast=0.5, w_adv=0.3), 'Full loss (published)'),
        ('mmd-huber', dict(w_contrast=0.0, w_adv=0.0), 'Huber + MMD only')]

adata_norm = prep.adata_base      # raw X + layers['normalized_base'] (Stage 1); arm layers are added here
base_log = np.log1p(np.clip(np.asarray(adata_norm.layers['normalized_base']), 0, None)).astype(np.float32)
batches_arr = adata_norm.obs['batch_id'].astype(str).values
LAYERS = [('normalized_base', 'Stage 1', None, None)]
histories, train_info = {}, {}


def matched_noise(delta, rng):
    # Stage 1 + zero-mean Gaussian noise, SD matched per batch and marker to `delta` (log1p)
    g = base_log.copy()
    for b in np.unique(batches_arr):
        ii = np.where(batches_arr == b)[0]
        g[ii] += rng.standard_normal((len(ii), delta.shape[1])).astype(np.float32) * delta[ii].std(axis=0)
    return np.clip(np.expm1(g), 0, None).astype(np.float32)


t0 = time.time()
for seed in SEEDS:
    for i, (arm, w, label) in enumerate(ARMS):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        t1 = time.time()
        model, hist = ss.train(prep, device_str=DEVICE, **BASE, **w)
        train_info[f'{arm}_s{seed}'] = (time.time() - t1) / 60
        delta = ss.correct(prep, model, device_str=DEVICE)
        key = f'{arm.replace("-", "_")}_s{seed}'
        adata_norm.layers[key] = ss.apply(prep, delta, alpha=ALPHA)
        LAYERS.append((key, f'{label} s{seed}', arm, seed))
        d_log = np.log1p(np.clip(adata_norm.layers[key], 0, None)) - base_log
        nkey = f'{arm.replace("-", "_")}_noise_s{seed}'
        adata_norm.layers[nkey] = matched_noise(d_log, np.random.default_rng(1000 * seed + 2 + i))
        LAYERS.append((nkey, f'{label} + matched noise s{seed}', f'{arm}-noise', seed))
        histories[f'{arm}_s{seed}'] = pd.DataFrame(hist)
        del model, delta, d_log
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'  {arm} s{seed}: train {train_info[f"{arm}_s{seed}"]:.1f} min '
              f'(total {(time.time() - t0) / 60:.1f} min)')

KBET_LAYERS = [k for k, _, a, _ in LAYERS if a is not None]
print(f'\n{len(LAYERS)} layers; final-epoch losses:')
for k, h in histories.items():
    print(f'  {k:14s} ' + '  '.join(f'{c} {h[c].iloc[-1]:.4f}' for c in ['loss', 'recon', 'contrast', 'adv', 'mmd']))

In [ ]:
# 3b. Divergence check — run after §3, before §4.
# Without the Huber term (no-Huber, all-off) nothing bounds the size of the correction, and the
# expm1 at inference overflows (max 4e10 / inf). Such layers break the metric cells (inf -> histogram
# and silhouette errors), so they are reported here and dropped from LAYERS: their verdict is
# "diverged". Criterion: any non-finite value, or > 0.1% of entries above 1.2x the raw per-marker
# log1p max (the same 1.2x cap rule as SpaNCy-GNN).
CAP = 1.2
raw_max = np.log1p(np.clip(X_raw, 0, None).max(axis=0))
diverged, div_rows = {}, []
for key, label, arm, seed in LAYERS:
    X = np.asarray(adata_norm.layers[key])
    with np.errstate(over='ignore', invalid='ignore'):
        X_log = np.log1p(np.clip(X, 0, None))
    finite = np.isfinite(X_log)
    over = ~finite | (X_log > CAP * raw_max)
    frac_over = over.mean(axis=0)
    bad_markers = [f'{marker_names[i]} {100 * frac_over[i]:.1f}%'
                   for i in np.argsort(-frac_over) if frac_over[i] > 0.001]
    is_div = (~finite).any() or over.mean() > 0.001
    if is_div and arm is not None:
        diverged[key] = (arm, seed)
    div_rows.append(dict(layer=label, nonfinite=int((~finite).sum()),
                         pct_over_cap=round(100 * float(over.mean()), 3),
                         max_log1p=round(float(X_log[finite].max()), 2),
                         diverged=is_div, markers_over_cap=', '.join(bad_markers[:6])))
    del X, X_log, finite, over
    gc.collect()

print(f'Raw log1p max over markers: {raw_max.max():.2f}  (cap = {CAP}x per marker)')
print(pd.DataFrame(div_rows).to_string(index=False))
LAYERS = [l for l in LAYERS if l[0] not in diverged]
print(f'\nDropped {len(diverged)} diverged layers from the metric cells: '
      f'{sorted(f"{a} s{s}" for a, s in diverged.values())}')

## 4. 1D shape + correction on bimodal markers

Same per-(marker, sample) ratios as cell 6b of the other notebooks (`_shape_ratios` copied verbatim),
but computed for **all** markers so the bimodal ones (ECAD, …) are visible. `bim_delta` = mean
|log1p(arm) − log1p(Stage 1)| over bimodal markers: ~0 means bimodal markers were left alone.

In [ ]:
N_SHAPE_BINS = 80
MIN_CELLS = 50
PEAK_LO, VAR_LO, VAR_HI = 0.80, 0.80, 1.25

sample_ids_shape = adata_norm.obs['sample_id'].values
sample_idx = {s: np.where(sample_ids_shape == s)[0] for s in np.unique(sample_ids_shape)}
sample_idx = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


shape_rows, shape_sum = [], {}
for key, label, arm, seed in LAYERS:
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        shape_rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame([r for r in shape_rows if r['key'] == key]).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )
    del X_log
    gc.collect()
shape_df = pd.DataFrame(shape_rows)

print('Shape summary (peak/var/iqr = unimodal means; dist = ALL markers incl. bimodal):')
print(pd.DataFrame(shape_sum).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 4b. Spread preservation — MAD (cyCombine-style), same definition as the region_batch run

`mad_rel` = median |MAD_after / MAD_before − 1| per cluster × sample, unimodal markers (Stage 1 = 0);
`mad_signed` < 0 = compression; `mad_cycombine` = cyCombine's exact score (cluster × batch, absolute).

In [ ]:
from sklearn.cluster import MiniBatchKMeans

N_CLUST = 10
MIN_GROUP = 50
_sub = np.random.default_rng(0).choice(len(base_log), min(200_000, len(base_log)), replace=False)
km = MiniBatchKMeans(n_clusters=N_CLUST, random_state=0, batch_size=10_000, n_init=3).fit(base_log[_sub])
clust = km.predict(base_log)
samp_m = adata_norm.obs['sample_id'].astype(str).values
bat_m = adata_norm.obs['batch_id'].astype(str).values
G_S = [ii for ii in pd.DataFrame({'c': clust, 'k': samp_m}).groupby(['c', 'k']).indices.values()
       if len(ii) >= MIN_GROUP]
G_B = [ii for ii in pd.DataFrame({'c': clust, 'k': bat_m}).groupby(['c', 'k']).indices.values()
       if len(ii) >= MIN_GROUP]


def _mad(X, groups):
    out = np.empty((len(groups), X.shape[1]), dtype=np.float64)
    for gi, ii in enumerate(groups):
        Xg = X[ii]
        out[gi] = np.median(np.abs(Xg - np.median(Xg, axis=0)), axis=0)
    return out


MAD0_S = _mad(X_raw_log, G_S)
MAD0_B = _mad(X_raw_log, G_B)
MAD_SUM = {}
for key, label, arm, seed in LAYERS:
    X = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    b = MAD0_S[:, uni_idx]
    a = _mad(X, G_S)[:, uni_idx]
    ok = b > 1e-3
    rel = a[ok] / b[ok] - 1
    MAD_SUM[key] = dict(mad_rel=float(np.median(np.abs(rel))), mad_signed=float(np.median(rel)),
                        mad_cycombine=float(np.median(np.abs(_mad(X, G_B) - MAD0_B))))
    del X
    gc.collect()
print(pd.DataFrame(MAD_SUM).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 5. Silhouette (all seeds)

In [ ]:
# Silhouette — UniFORM-style, 3 cell types, PER SAMPLE (same method + guard as the other notebooks).
N_SIL = 3000
MIN_TYPE_CELLS = 50

anchor_thresholds = {}
for mname in ['ECAD', 'CD45', 'aSMA']:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(X_log_preview[:, mi].reshape(-1, 1))
    anchor_thresholds[mname] = float(np.mean(gmm.means_))
print('Anchor GMM thresholds (log1p, raw):', {k: round(v, 3) for k, v in anchor_thresholds.items()})

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']] = 2
cell_type[X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']] = 1
cell_type[X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']] = 0

# Per-sample selection is decided once (deterministic guard, rng 42 per sample as before).
sample_ids_all = adata_norm.obs['sample_id'].values
sil_sel, dropped = {}, []
for sample in sorted(np.unique(sample_ids_all)):
    smask_idx = np.where(sample_ids_all == sample)[0]
    s_labels = cell_type[smask_idx]
    big = [c for c in (0, 1, 2) if int((s_labels == c).sum()) >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sil_sel[sample] = (smask_idx[sel], s_labels[sel])

sil_rows = {}
for key in ['raw'] + [k for k, _, _, _ in LAYERS]:
    X = X_raw if key == 'raw' else np.asarray(adata_norm.layers[key])
    sil_rows[key] = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
                     for s, (ii, lab) in sil_sel.items()}
sil_df = pd.DataFrame(sil_rows)
sil_df.loc['Mean'] = sil_df.mean()
print(f'{len(sil_sel)} samples passed the guard (dropped: {dropped})')
print(sil_df.loc[['Mean']].round(4).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).to_string())

## 6. Positive population (`POP_SEEDS`)

In [ ]:
pop_sum = {}
for key, label, arm, seed in LAYERS:
    if seed is not None and seed not in POP_SEEDS:
        continue
    pt = positive_population_table(adata_norm, norm_layer=key, sample_col='sample_id')
    s = pt.groupby('marker')['delta'].mean()
    pop_sum[key] = dict(pass_5pct=int((s.abs() < 5).sum()),
                        mean_abs_delta=float(pt['delta'].abs().mean()))
    print(f'  {label:16s} {pop_sum[key]["pass_5pct"]}/20 within +/-5%   '
          f'mean |delta| {pop_sum[key]["mean_abs_delta"]:.2f}%')

## 7. Histograms — Raw | Stage 1 | full loss | Huber + MMD only (seed 6)

Per-sample overlay curves (one line per sample, tab20) in log1p space — same cell as
`spancy_shift_region_final.ipynb`, only the layer list, folder and file name changed. Inline: CD3,
CD31, HLADRB1, aSMA (corrected by Stage 2) + ECAD (bimodal, must look exactly like Stage 1).
Full PDF: all 20 markers.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
import os
import shutil

HIST_DIR = 'histograms_lossterms'
WATCH = ['CD3', 'CD31', 'HLADRB1', 'aSMA', 'ECAD']
N_HIST_BINS = 100
MARKERS_PER_PAGE = 4
RAW = '__RAW__'
os.makedirs(HIST_DIR, exist_ok=True)
HIST_LAYERS = [(RAW, 'Raw'), ('normalized_base', 'Stage 1'), ('full_s6', 'Full loss (published)'),
               ('mmd_huber_s6', 'Huber + MMD only')]
HIST_LAYERS = [(k, l) for k, l in HIST_LAYERS if k == RAW or k in adata_norm.layers]

M_h = len(marker_names)
sample_ids_h = adata_norm.obs['sample_id'].values
samples_h = sorted(np.unique(sample_ids_h).tolist())
sidx_h = {s: np.where(sample_ids_h == s)[0] for s in samples_h}
colors_h = plt.cm.tab20(np.linspace(0, 1, len(samples_h)))
edges_h = {}
for mi in range(M_h):
    v = X_raw_log[:, mi]
    hi = float(np.percentile(v, 99.9))
    edges_h[mi] = np.linspace(float(v.min()), hi if hi > v.min() else float(v.max()), N_HIST_BINS + 1)

dens_h = {}
for key, _ in HIST_LAYERS:
    XL = X_raw_log if key == RAW else np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi in range(M_h):
        for s in samples_h:
            dens_h[(key, mi, s)], _ = np.histogram(XL[sidx_h[s], mi], bins=edges_h[mi], density=True)
    del XL
    gc.collect()
ymax_h = {mi: max(d.max() for (k, mm, s), d in dens_h.items() if mm == mi) for mi in range(M_h)}


def _plot_rows(marker_idx):
    fig, axes = plt.subplots(len(marker_idx), len(HIST_LAYERS),
                             figsize=(2.9 * len(HIST_LAYERS), 2.3 * len(marker_idx)), squeeze=False, sharex='row')
    for r, mi in enumerate(marker_idx):
        ctr = 0.5 * (edges_h[mi][:-1] + edges_h[mi][1:])
        for c, (key, label) in enumerate(HIST_LAYERS):
            ax = axes[r, c]
            for s, col in zip(samples_h, colors_h):
                ax.plot(ctr, dens_h[(key, mi, s)], color=col, lw=0.8)
            ax.set_ylim(0, ymax_h[mi] * 1.05)
            if r == 0:
                ax.set_title(label, fontsize=9)
            if c == 0:
                ax.set_ylabel(marker_names[mi], fontsize=9)
            ax.tick_params(labelsize=7)
    fig.tight_layout()
    return fig


_plot_rows([marker_names.index(n) for n in WATCH if n in marker_names])
plt.show()
pdf_path = os.path.join(HIST_DIR, 'lossterms_histograms.pdf')
with PdfPages(pdf_path) as pdf:
    for p in range(0, M_h, MARKERS_PER_PAGE):
        fig = _plot_rows(list(range(p, min(p + MARKERS_PER_PAGE, M_h))))
        pdf.savefig(fig, dpi=150)
        plt.close(fig)
print(f'Saved {pdf_path}')
try:
    from google.colab import files as _files
    _files.download(pdf_path)
except Exception as e:
    print(f'(download skipped: {e})')

## 7c. Save for a later kBET run (download)

In [ ]:
SAVE_FOR_LATER = True
if SAVE_FOR_LATER:
    import os, pickle, shutil
    from google.colab import files
    SAVE_DIR = '/content/lossterms_save'
    os.makedirs(SAVE_DIR, exist_ok=True)
    kb_samples = sorted({s for pairs in GROUP_DEFS.values() for s, _ in pairs})
    rows = np.where(adata_norm.obs['sample_id'].astype(str).isin(kb_samples).values)[0]
    adata_norm.obs.iloc[rows].to_pickle(f'{SAVE_DIR}/obs.pkl')
    adata_norm.var.to_pickle(f'{SAVE_DIR}/var.pkl')
    for k in KBET_LAYERS:
        np.save(f'{SAVE_DIR}/{k}.npy', np.asarray(adata_norm.layers[k], dtype=np.float32)[rows])
    state = dict(SEEDS=SEEDS, LAYERS=LAYERS, KBET_LAYERS=KBET_LAYERS, GROUP_DEFS=GROUP_DEFS,
                 shape_sum=shape_sum, MAD_SUM=MAD_SUM, sil_df=sil_df, pop_sum=pop_sum,
                 diverged=diverged, train_info=train_info, histories=histories)
    with open(f'{SAVE_DIR}/state.pkl', 'wb') as f:
        pickle.dump(state, f)
    shutil.make_archive('/content/lossterms_save', 'zip', SAVE_DIR)
    print(f'{len(rows):,} cells, {len(KBET_LAYERS)} layers, '
          f'zip {os.path.getsize("/content/lossterms_save.zip") / 1e6:.0f} MB')
    files.download('/content/lossterms_save.zip')

## 8. kBET — 2 arms + 2 matched-noise controls × 3 seeds = 12 layers (≈ 4 h; can run later from the §7c zip)
**To stop before kBET:** click this header → Runtime → Run before.

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

ABL_LABEL = {k: l for k, l, _, _ in LAYERS}
EVAL_LAYERS = KBET_LAYERS
print(f'kBET on {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

## 9. Result (paste this)

In [ ]:
all_kbet = globals().get('all_kbet', {})   # §9 also runs (without kBET) if §8 was skipped


def _kbet(k):
    if k not in all_kbet:
        return np.nan
    kv = [x['kBET'] for x in all_kbet[k].values() if not np.isnan(x['kBET'])]
    return float(np.mean(kv)) if kv else np.nan


recs = []
for key, label, arm, seed in LAYERS:
    if arm is None:
        continue
    r = dict(arm=arm, seed=seed, kBET=_kbet(key),
             sil=float(sil_df.loc['Mean', key]) if key in sil_df.columns else np.nan)
    if key in MAD_SUM:
        r.update(mad_rel=MAD_SUM[key]['mad_rel'], mad_signed=MAD_SUM[key]['mad_signed'])
    if key in shape_sum:
        r.update(dist=shape_sum[key]['dist'], var=shape_sum[key]['var'], peak=shape_sum[key]['peak'],
                 iqr=shape_sum[key]['iqr'], uni_delta=shape_sum[key]['uni_delta'],
                 bim_delta=shape_sum[key]['bim_delta'], dist_list=shape_sum[key]['dist_list'])
    if key in pop_sum:
        r['pos_pop'] = pop_sum[key]['pass_5pct']
    recs.append(r)
res = pd.DataFrame(recs)
cols = [c for c in ['kBET', 'sil', 'mad_rel', 'mad_signed', 'dist', 'var', 'peak', 'iqr', 'uni_delta',
                    'bim_delta', 'pos_pop'] if c in res]
print('=' * 100)
print('LOSS-TERM TEST — seeds %s, alpha 0.6 (Stage 1: silhouette %.4f)'
      % (SEEDS, float(sil_df.loc['Mean', 'normalized_base'])))
print('=' * 100)
print(res.groupby('arm', sort=False)[cols].agg(['mean', 'std']).round(4).to_string())
print('\nPer seed:')
print(res.drop(columns=['dist_list'], errors='ignore').round(4).to_string(index=False))
print('\nDistorted markers (old shape rule, all markers):')
for _, r in res.iterrows():
    d = int(r['dist']) if pd.notna(r.get('dist')) else 'NA'
    print(f'  {r.arm:16s} s{r.seed}: {d}  {r.get("dist_list", "")}')
print('\nPer-group kBET:')
for k in KBET_LAYERS:
    if k in all_kbet:
        print(f'  {k:22s} ' + '  '.join(f'{g} {v["kBET"]:.3f}' for g, v in all_kbet[k].items()))

P = res.set_index(['arm', 'seed'])


def v(arm, col):
    return np.array([P.loc[(arm, s), col] for s in SEEDS], dtype=float)


dk = v('mmd-huber', 'kBET') - v('full', 'kBET')
ds = v('mmd-huber', 'sil') - v('full', 'sil')
ddist = v('mmd-huber', 'dist') - v('full', 'dist')
dpop = v('mmd-huber', 'pos_pop') - v('full', 'pos_pop')
print('\n' + '=' * 100)
print('PAIRED mmd-huber - full (per seed %s)' % SEEDS)
print('=' * 100)
for name, d in [('kBET', dk), ('silhouette', ds), ('distorted', ddist), ('pos-pop pass', dpop)]:
    print(f'  {name:13s} ' + '  '.join(f's{s} {x:+.4f}' for s, x in zip(SEEDS, d)) + f'   mean {np.mean(d):+.4f}')
for arm in ['full', 'mmd-huber']:
    sk = v(arm, 'kBET') - v(f'{arm}-noise', 'kBET')
    ss_ = v(arm, 'sil') - v(f'{arm}-noise', 'sil')
    print(f'  {arm:10s} minus own noise: kBET ' + ' '.join(f'{x:+.4f}' for x in sk)
          + '   sil ' + ' '.join(f'{x:+.4f}' for x in ss_)
          + f'   (correction size {np.mean(v(arm, "uni_delta")):.4f})')

kb_mh, kb_full = v('mmd-huber', 'kBET'), v('full', 'kBET')
have_kbet = not (np.isnan(kb_mh).any() or np.isnan(kb_full).any()
                 or np.isnan(v('mmd-huber-noise', 'kBET')).any())
R1 = bool(np.mean(dk) >= -0.011) if have_kbet else None
R2 = bool(np.mean(ds) >= -0.002 and ds.min() >= -0.005 and (ddist <= 0).all() and (dpop >= 0).all())
R3 = (bool((kb_mh > v('mmd-huber-noise', 'kBET')).all() and (v('mmd-huber', 'sil') > v('mmd-huber-noise', 'sil')).all())
      if have_kbet else None)
R4 = bool(np.mean(kb_mh) >= 0.7016 and (v('mmd-huber', 'sil') > 0.3288).all()) if have_kbet else None
R5 = bool(np.std(kb_mh, ddof=1) <= 2 * np.std(kb_full, ddof=1)) if have_kbet else None
print('\n' + '=' * 100)
print('ADOPTION RULE (fixed before the run)')
print('=' * 100)
for tag, ok, text in [
        ('1', R1, f'kBET no worse: mean dkBET {np.mean(dk):+.4f} >= -0.011'),
        ('2', R2, f'biology no worse: mean dsil {np.mean(ds):+.4f} >= -0.002, min {ds.min():+.4f} >= -0.005, '
                  f'distorted not up, pos-pop not down'),
        ('3', R3, 'C1: kBET and silhouette above own matched noise at every seed'),
        ('4', R4, f'C2: mean kBET {np.mean(kb_mh):.4f} >= 0.7016 and silhouette > 0.3288 every seed'),
        ('5', R5, f'stability: kBET SD {np.std(kb_mh, ddof=1):.4f} <= 2 x full {np.std(kb_full, ddof=1):.4f}')]:
    mark = '?' if ok is None else ('x' if ok else ' ')
    print(f'  [{mark}] {tag}. {text}: {ok}')
rules = [R1, R2, R3, R4, R5]
if None in rules:
    print('\n  VERDICT: UNDECIDED — run §8 kBET first')
elif all(rules):
    print('\n  VERDICT: ADOPT Huber + MMD (all five criteria pass)')
else:
    print('\n  VERDICT: KEEP the published loss (criteria failed: '
          + ', '.join(str(i + 1) for i, ok in enumerate(rules) if not ok) + ')')
print('\nReference (spancy_shift_region_final, seeds 6-8): region GNN kBET 0.704 +/- 0.011, silhouette 0.358 +/- 0.001')

## R. Restore in a FRESH runtime (only for a later kBET run)
Run the §0 install cell, then this cell (`RESTORE = True`), then §8 (`sec8-setup`, `sec8-kbet`) and §9.

In [ ]:
RESTORE = False
if RESTORE:
    import os, gc, zipfile, pickle
    import numpy as np, pandas as pd, anndata as ad
    from google.colab import files
    if not os.path.exists('lossterms_save.zip'):
        files.upload()
    zipfile.ZipFile('lossterms_save.zip').extractall('lossterms_save')
    D = 'lossterms_save'
    with open(f'{D}/state.pkl', 'rb') as f:
        globals().update(pickle.load(f))
    adata_norm = ad.AnnData(obs=pd.read_pickle(f'{D}/obs.pkl'), var=pd.read_pickle(f'{D}/var.pkl'))
    for k in KBET_LAYERS:
        adata_norm.layers[k] = np.load(f'{D}/{k}.npy')
    print(f'Restored {adata_norm.n_obs:,} cells, kBET layers: {KBET_LAYERS}')